# 05 - Clasificacion Multiclase: Tipo de Viaje

## Pregunta de negocio: Podemos clasificar automaticamente el tipo de viaje?

En este notebook construiremos un clasificador multiclase para categorizar automaticamente
los viajes de taxi en tipos significativos para el negocio:

| Tipo | Descripcion |
|------|-------------|
| **airport** | Viajes hacia/desde JFK o LaGuardia |
| **nocturno** | Viajes entre las 22:00 y las 05:00 |
| **commute** | Viajes laborales en hora pico |
| **turistico** | Viajes de fin de semana en Manhattan |
| **corto** | Viajes de menos de 1 milla |

### Modelos a explorar:
- One-vs-Rest con Logistic Regression
- Random Forest (nativo multiclase)
- XGBoost (nativo multiclase)

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

## 1. Carga de datos y creacion del target multiclase

In [ ]:
# Imports principales
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuracion
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Imports cargados correctamente')

In [ ]:
# Consulta con campos necesarios para clasificar tipo de viaje
query = """
SELECT
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS pickup_dow,
    EXTRACT(MONTH FROM pickup_datetime) AS pickup_month,
    trip_distance,
    passenger_count,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    rate_code,
    pickup_location_id,
    dropoff_location_id,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) AS trip_duration_min,
    CASE
        WHEN trip_distance > 0 AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) > 0
        THEN trip_distance / (TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) / 60.0)
        ELSE 0
    END AS avg_speed_mph
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    fare_amount > 2.5
    AND fare_amount < 200
    AND trip_distance > 0
    AND trip_distance < 50
    AND total_amount > 0
    AND passenger_count > 0
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) BETWEEN 1 AND 180
ORDER BY RAND()
LIMIT 200000
"""

df = bq.query_to_dataframe(query)
print(f'Registros cargados: {len(df):,}')
df.head()

In [ ]:
# Crear target multiclase basado en caracteristicas del viaje
# Prioridad de reglas: airport > nocturno > commute > turistico > corto

def classify_trip(row):
    """
    Clasifica un viaje en una de 5 categorias basado en sus caracteristicas.
    Se aplican reglas con prioridad para evitar solapamiento.
    """
    hour = row['pickup_hour']
    dow = row['pickup_dow']  # 1=Dom, 2=Lun, ..., 7=Sab
    distance = row['trip_distance']
    rate = row['rate_code']
    speed = row['avg_speed_mph']
    
    # 1. AIRPORT: rate_code 2 (JFK) o 3 (Newark) o viajes largos con peaje
    if rate in [2, 3]:
        return 'airport'
    
    # 2. NOCTURNO: pickup entre 22:00 y 05:00
    if hour >= 22 or hour < 5:
        return 'nocturno'
    
    # 3. COMMUTE: dias laborales, hora pico, distancia corta-media
    is_weekday = dow in [2, 3, 4, 5, 6]  # Lunes a Viernes
    is_rush_hour = (7 <= hour <= 9) or (17 <= hour <= 19)
    if is_weekday and is_rush_hour and 1 <= distance <= 10:
        return 'commute'
    
    # 4. TURISTICO: fin de semana, distancia media, velocidad baja (trafico turistico)
    is_weekend = dow in [1, 7]  # Domingo o Sabado
    if is_weekend and 1 <= distance <= 8 and speed < 15:
        return 'turistico'
    
    # 5. CORTO: distancia menor a 1 milla
    if distance < 1:
        return 'corto'
    
    # Default: si no entra en ninguna categoria, se asigna segun la mas cercana
    if distance < 3:
        return 'corto'
    elif is_weekday:
        return 'commute'
    else:
        return 'turistico'


df['trip_type'] = df.apply(classify_trip, axis=1)
print('Tipos de viaje creados:')
print(df['trip_type'].value_counts())

## 2. Reglas de priorizacion y solapamiento

Las reglas se aplican con **prioridad estricta** para evitar que un viaje pertenezca
a multiples categorias:

1. **airport** (maxima prioridad): identificado por rate_code, no ambiguo
2. **nocturno**: horario nocturno, incluso si coincide con otras categorias
3. **commute**: dia laboral + hora pico, solo si no es nocturno ni aeropuerto
4. **turistico**: fin de semana + patron turistico
5. **corto** (menor prioridad): distancia corta, categoria "residual"

Esta jerarquia refleja que los viajes al aeropuerto y nocturnos tienen
caracteristicas operativas mas distintivas y relevantes para el negocio.

## 3. Visualizacion de la distribucion de clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Conteo de clases
type_counts = df['trip_type'].value_counts()
type_pcts = df['trip_type'].value_counts(normalize=True) * 100

colors_map = {
    'commute': '#3498db',
    'corto': '#2ecc71',
    'nocturno': '#9b59b6',
    'turistico': '#e67e22',
    'airport': '#e74c3c'
}
bar_colors = [colors_map.get(t, '#95a5a6') for t in type_counts.index]

# Grafico de barras
bars = axes[0].bar(type_counts.index, type_counts.values, color=bar_colors, edgecolor='black')
for bar, count, pct in zip(bars, type_counts.values, type_pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{count:,}\n({pct:.1f}%)', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Distribucion de Tipos de Viaje', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Cantidad de viajes')

# Grafico de pastel
axes[1].pie(type_counts.values, labels=type_counts.index, colors=bar_colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporcion de Tipos de Viaje', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

# Estadisticas por tipo
print('\nEstadisticas por tipo de viaje:')
print(df.groupby('trip_type').agg({
    'trip_distance': 'mean',
    'fare_amount': 'mean',
    'trip_duration_min': 'mean',
    'avg_speed_mph': 'mean'
}).round(2).to_string())

In [ ]:
# Preparar datos para modelado
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Features
feature_cols = [
    'pickup_hour', 'pickup_dow', 'pickup_month',
    'trip_distance', 'passenger_count', 'fare_amount',
    'tip_amount', 'tolls_amount', 'trip_duration_min', 'avg_speed_mph'
]

X = df[feature_cols].fillna(0).copy()
le = LabelEncoder()
y = le.fit_transform(df['trip_type'])

print('Mapeo de clases:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} -> {cls}')

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar para Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTrain: {len(X_train):,} muestras')
print(f'Test:  {len(X_test):,} muestras')
print(f'Numero de clases: {len(le.classes_)}')

## 4. Modelo 1: OneVsRest con Regresion Logistica

La estrategia **One-vs-Rest (OvR)** descompone un problema multiclase en
N problemas binarios (uno por clase). Para cada clase, entrena un clasificador
que distingue esa clase de todas las demas.

**Ventaja:** Permite usar cualquier clasificador binario para problemas multiclase.

**Desventaja:** No captura relaciones entre clases.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# OneVsRest con Logistic Regression
ovr_model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
)
ovr_model.fit(X_train_scaled, y_train)

y_pred_ovr = ovr_model.predict(X_test_scaled)

print('=== One-vs-Rest con Logistic Regression ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_ovr):.4f}')
print()
print(classification_report(y_test, y_pred_ovr, target_names=le.classes_))

## 5. Modelo 2: Random Forest (multiclase nativo)

Random Forest maneja multiclase de forma nativa, sin necesidad de estrategias
como OvR. Cada arbol vota por una clase y la clase con mas votos gana.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [10, 20],
    'class_weight': ['balanced']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, cv=3, scoring='f1_macro',
    n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)

rf_model = rf_grid.best_estimator_
y_pred_rf = rf_model.predict(X_test)

print(f'\nMejores hiperparametros: {rf_grid.best_params_}')
print(f'Mejor F1 macro (CV): {rf_grid.best_score_:.4f}')
print()
print('=== Random Forest Multiclase ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

## 6. Modelo 3: XGBoost multiclase

XGBoost usa `multi:softmax` o `multi:softprob` para clasificacion multiclase.
Internamente, entrena N arboles por iteracion (uno por clase) y usa softmax
para convertir los outputs en probabilidades.

In [ ]:
from xgboost import XGBClassifier

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

xgb_grid = GridSearchCV(
    XGBClassifier(
        objective='multi:softprob',
        num_class=len(le.classes_),
        random_state=42,
        eval_metric='mlogloss',
        use_label_encoder=False
    ),
    xgb_params, cv=3, scoring='f1_macro',
    n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train, y_train)

xgb_model = xgb_grid.best_estimator_
y_pred_xgb = xgb_model.predict(X_test)

print(f'\nMejores hiperparametros: {xgb_grid.best_params_}')
print(f'Mejor F1 macro (CV): {xgb_grid.best_score_:.4f}')
print()
print('=== XGBoost Multiclase ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

## 7. Matrices de confusion para cada modelo

La matriz de confusion multiclase muestra donde cada modelo acierta y donde se confunde.
Los valores en la diagonal son clasificaciones correctas.

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

models_preds = {
    'OvR Logistic Regression': y_pred_ovr,
    'Random Forest': y_pred_rf,
    'XGBoost': y_pred_xgb
}
cmaps = ['Blues', 'Greens', 'Oranges']

for ax, (name, y_pred), cmap in zip(axes, models_preds.items(), cmaps):
    cm = confusion_matrix(y_test, y_pred)
    # Normalizar por fila para mostrar porcentajes
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Anotaciones: conteo y porcentaje
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f'{cm[i,j]:,}\n({cm_norm[i,j]:.1%})'
    
    sns.heatmap(cm_norm.astype(float), annot=annot, fmt='', cmap=cmap,                xticklabels=le.classes_, yticklabels=le.classes_,
                ax=ax, vmin=0, vmax=1)
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_xlabel('Prediccion')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()

## 8. Metricas por clase: Precision, Recall y F1 para cada tipo de viaje

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Metricas por clase para cada modelo
per_class_metrics = []

for name, y_pred in models_preds.items():
    prec = precision_score(y_test, y_pred, average=None, labels=range(len(le.classes_)))
    rec = recall_score(y_test, y_pred, average=None, labels=range(len(le.classes_)))
    f1 = f1_score(y_test, y_pred, average=None, labels=range(len(le.classes_)))
    
    for i, cls in enumerate(le.classes_):
        per_class_metrics.append({
            'Modelo': name,
            'Tipo Viaje': cls,
            'Precision': prec[i],
            'Recall': rec[i],
            'F1': f1[i]
        })

metrics_df = pd.DataFrame(per_class_metrics)

# Visualizar F1 por clase y modelo
fig, ax = plt.subplots(figsize=(14, 6))
pivot_f1 = metrics_df.pivot(index='Tipo Viaje', columns='Modelo', values='F1')
pivot_f1.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('F1-Score por Tipo de Viaje y Modelo', fontweight='bold', fontsize=14)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.1)
ax.legend(title='Modelo', bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Tabla detallada
print('Metricas detalladas por clase (mejor modelo: XGBoost):')
xgb_metrics = metrics_df[metrics_df['Modelo'] == 'XGBoost'].set_index('Tipo Viaje')
print(xgb_metrics[['Precision', 'Recall', 'F1']].round(4).to_string())

## 9. F1 Macro vs F1 Micro: cuando usar cada uno

- **F1 Micro:** Calcula las metricas globalmente (suma TP, FP, FN de todas las clases).
  Favorece a las clases con mas muestras. Equivalente a accuracy en multiclase.

- **F1 Macro:** Promedio simple del F1 de cada clase. Trata todas las clases por igual,
  sin importar su tamano. Mas sensible al rendimiento en clases minoritarias.

**Recomendacion:**
- Usar **macro** cuando todas las clases son igualmente importantes
- Usar **micro** cuando las clases mayoritarias importan mas
- Si hay desbalance severo, **macro** revela mejor los problemas

In [ ]:
# Comparar F1 macro vs micro para cada modelo
print('='*60)
print('COMPARACION F1 MACRO vs F1 MICRO')
print('='*60)

comparison_data = []
for name, y_pred in models_preds.items():
    f1_mac = f1_score(y_test, y_pred, average='macro')
    f1_mic = f1_score(y_test, y_pred, average='micro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    comparison_data.append({
        'Modelo': name,
        'F1 Macro': f1_mac,
        'F1 Micro': f1_mic,
        'F1 Weighted': f1_weighted,
        'Diferencia (Micro-Macro)': f1_mic - f1_mac
    })

comp_df = pd.DataFrame(comparison_data).set_index('Modelo')
print(comp_df.round(4).to_string())

print('\nInterpretacion:')
print('- Si Micro >> Macro: el modelo rinde bien en clases grandes pero mal en pequenas')
print('- Si Micro ~ Macro: el rendimiento es uniforme entre clases')
print('- F1 Weighted pondera por tamano de clase (compromiso entre macro y micro)')

## 10. Analisis de errores: cuales tipos se confunden mas?

In [ ]:
# Analisis detallado de confusiones del mejor modelo (XGBoost)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

# Encontrar los pares mas confundidos
confusions = []
for i in range(len(le.classes_)):
    for j in range(len(le.classes_)):
        if i != j and cm_xgb[i, j] > 0:
            confusions.append({
                'Real': le.classes_[i],
                'Predicho': le.classes_[j],
                'Cantidad': cm_xgb[i, j],
                'Porcentaje': cm_xgb[i, j] / cm_xgb[i].sum() * 100
            })

conf_df = pd.DataFrame(confusions).sort_values('Cantidad', ascending=False)

print('='*60)
print('TOP 10 CONFUSIONES MAS FRECUENTES (XGBoost)')
print('='*60)
print(conf_df.head(10).to_string(index=False))

# Visualizar las confusiones principales
fig, ax = plt.subplots(figsize=(12, 6))
top_conf = conf_df.head(8).copy()
top_conf['Par'] = top_conf['Real'] + ' -> ' + top_conf['Predicho']
ax.barh(top_conf['Par'], top_conf['Cantidad'], color='#e74c3c', edgecolor='black')
ax.set_xlabel('Cantidad de errores')
ax.set_title('Principales Confusiones del Modelo XGBoost', fontweight='bold', fontsize=13)

# Anotar porcentajes
for i, (count, pct) in enumerate(zip(top_conf['Cantidad'], top_conf['Porcentaje'])):
    ax.text(count + 10, i, f'{pct:.1f}% de la clase real', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\nAnalisis:')
print('Las confusiones mas comunes tipicamente ocurren entre:')
print('- commute <-> corto: viajes laborales cortos se confunden con viajes cortos generales')
print('- turistico <-> commute: ambos pueden tener distancias similares')
print('- airport es generalmente el tipo mas facil de clasificar (rate_code distintivo)')

## 11. Guardar el mejor modelo

In [ ]:
import joblib
import os

# Crear directorio si no existe
model_dir = '../../../models/nyc_taxi/'
os.makedirs(model_dir, exist_ok=True)

# Determinar mejor modelo por F1 macro
model_scores = {}
models_map = {
    'OvR Logistic Regression': ovr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}

for name, y_pred in models_preds.items():
    model_scores[name] = f1_score(y_test, y_pred, average='macro')

best_model_name = max(model_scores, key=model_scores.get)
best_model = models_map[best_model_name]

# Guardar modelo con metadata
model_artifact = {
    'model': best_model,
    'scaler': scaler,
    'label_encoder': le,
    'feature_cols': feature_cols,
    'model_name': best_model_name,
    'class_names': list(le.classes_),
    'f1_macro': model_scores[best_model_name]
}

model_path = os.path.join(model_dir, 'trip_type_classifier.joblib')
joblib.dump(model_artifact, model_path)

print(f'Modelo guardado en: {model_path}')
print(f'Modelo seleccionado: {best_model_name}')
print(f'F1 Macro: {model_scores[best_model_name]:.4f}')
print(f'Tamano del archivo: {os.path.getsize(model_path) / 1024:.1f} KB')
print(f'\nClases soportadas: {list(le.classes_)}')

## Conclusiones

### Hallazgos principales:

1. **Tipos de viaje bien diferenciados:** Los viajes al aeropuerto son los mas faciles
   de clasificar gracias al rate_code distintivo, mientras que los viajes cortos y commute
   son mas dificiles de separar.

2. **Random Forest y XGBoost superan a OvR:** Los modelos de ensamble capturan
   interacciones entre features que la regresion logistica no puede modelar.

3. **F1 Macro es la metrica clave:** Dado que queremos buen rendimiento en todas
   las categorias, F1 macro es mas informativo que accuracy.

4. **Confusiones esperadas:** Los pares mas confundidos reflejan solapamiento real
   en las caracteristicas de ciertos tipos de viaje.

### Proximos pasos:
- Interpretar los modelos para entender que features impulsan cada clasificacion (notebook 06)
- Considerar features adicionales (clima, eventos, zonas mas granulares)